# Handoff 상태 전환

대화의 담당자가 바뀌어도 다음 턴이 같은 담당자에서 이어지게 하려면 어떤 상태를 저장해야 할까요?


## 실행 환경 준비

이 노트북은 로컬 StateGraph와 메모리 체크포인터만 사용합니다. 모델 API 키는 필요하지 않습니다.


In [1]:
# 로컬 그래프 실행에 필요한 패키지만 설치합니다.
%pip install -q langgraph==1.2.11
from importlib.metadata import version

print({"langgraph": version("langgraph"), "execution": "local graph only"})


Note: you may need to restart the kernel to use updated packages.
{'langgraph': '1.2.11', 'execution': 'local graph only'}


## 활성 담당자 상태 정의

### 코드 읽기와 비교 경계

`active_agent`는 현재 대화를 담당하는 이름을 상태에 저장하는 필드입니다. 첫 호출은 `message`, 빈 `active_agent`, 빈 `reply`를 넣지만 두 번째 호출은 새 `message`만 전달합니다. 같은 `thread_id`와 체크포인터를 사용하므로 런타임이 직전 상태를 읽고 누락된 `active_agent`를 이어 줍니다. `dispatch()`는 새 메시지에 명시적인 전환 신호가 있으면 담당자를 바꾸고, 그렇지 않으면 저장된 담당자를 유지합니다. `respond()`는 갱신된 상태를 받아 답변을 만듭니다.

핸드오프는 함수를 한 번 호출하고 결과를 반환받는 서브에이전트 위임과 다릅니다. 위임 뒤에는 중앙 에이전트가 계속 사용자와 대화하지만, 핸드오프 뒤에는 선택된 담당자가 다음 턴의 제어권을 가집니다. 배송 문의가 해결된 뒤 일반 상담으로 복귀하려면 `active_agent`를 언제 초기화할지 별도 전환 규칙이 필요합니다. 이 예제의 종료 판정은 주문 번호 문자열과 담당자 이름만 확인하므로 실제 주문이 존재하거나 환불이 가능한지는 검증하지 않습니다.


In [2]:
# 여러 턴에 유지할 active_agent 필드를 상태에 정의합니다.
from typing import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver

class State(TypedDict):
    message: str
    active_agent: str
    reply: str


## 담당자 선택과 응답 함수

`dispatch()`는 현재 메시지와 저장된 담당자를 함께 읽고 새 `active_agent`만 반환합니다. `respond()`는 이 갱신된 상태를 받아 답변을 만듭니다.


In [3]:
# 메시지와 이전 상태를 읽어 담당자를 선택하고 답변을 만듭니다.
def dispatch(state: State) -> dict[str, str]:
    text = state["message"]
    if "환불" in text:
        active = "refund_agent"
    elif state.get("active_agent"):
        active = state["active_agent"]
    else:
        active = "general_agent"
    return {"active_agent": active}

def respond(state: State) -> dict[str, str]:
    return {"reply": f"{state['active_agent']}가 답변합니다: {state['message']}"}


## 체크포인터가 있는 그래프


In [4]:
# 체크포인터를 연결해 같은 thread_id의 상태를 다음 턴으로 이어갑니다.
builder = StateGraph(State)
builder.add_node("dispatch", dispatch)
builder.add_node("respond", respond)
builder.add_edge(START, "dispatch")
builder.add_edge("dispatch", "respond")
builder.add_edge("respond", END)
graph = builder.compile(checkpointer=InMemorySaver())


## 두 턴을 같은 thread에서 실행

첫 호출과 둘째 호출이 같은 `thread_id`를 써야 체크포인터가 이전 담당자를 찾을 수 있습니다. 다른 ID를 쓰면 독립된 대화로 처리됩니다.


In [5]:
# 첫 턴에서 담당자를 바꾸고 두 번째 턴에서 유지되는지 확인합니다.
config = {"configurable": {"thread_id": "handoff-demo"}}
turn1 = graph.invoke({"message": "환불 담당자에게 연결해 주세요", "active_agent": "", "reply": ""}, config)
turn2 = graph.invoke({"message": "주문 번호는 A-102입니다"}, config)
print(turn1["active_agent"], "→", turn2["active_agent"])
print(turn2["reply"])
assert turn2["active_agent"] == "refund_agent"


refund_agent → refund_agent
refund_agent가 답변합니다: 주문 번호는 A-102입니다


## 종료 조건 확인


In [6]:
# 필수 주문 번호와 담당자 상태가 모두 갖춰졌는지 판정합니다.
completed = "A-102" in turn2["message"] and turn2["active_agent"] == "refund_agent"
print({"필수정보_수집": completed, "다음_상태": "END" if completed else "refund_agent"})
assert completed


{'필수정보_수집': True, '다음_상태': 'END'}
